# 第 6 章｜RBAC + AES-256-GCM

依序執行每一格；可修改標示的參數後重跑。

## 執行前：可替換設定總覽

以下項目都可以依測試環境替換：

- `.env` Provider：填 `CILLM_API_KEY` 會走 CILLM；只有 `OPENAI_API_KEY` 時會走 OpenAI；兩者都有時 CILLM 優先。
- CILLM：`CILLM_BASE_URL`、`CILLM_USER_ID`、`CILLM_PLATFORM`、`CILLM_AGENT`、`GPT_OSS_MODEL_NAME`、`GEMMA_MODEL_NAME`。
- OpenAI：`OPENAI_API_KEY`、`OPENAI_MODEL_NAME`；本教材預設 `gpt-4o`，會同時模擬 GPT-OSS 文字與 Gemma 圖片流程。
- 密碼式 AES：第 6 章由使用者在 Notebook 隱藏輸入設定保險庫密碼；密碼不寫入 `.env` 或加密檔。
- 路徑：只有從其他工作目錄啟動 Notebook 時才需要調整 `ROOT`；一般從教材根目錄或 `notebooks/` 啟動不必修改。

> 請勿把含有真實 Key 的 `.env`、Notebook 輸出或截圖提交到 Git。

### 本章可替換

- `CURRENT_ROLE`：替換為 `guest`、`employee`、`operations`、`maintenance`、`developer` 或 `admin`。
- `wanted_tool`：替換要檢查的 Tool，例如 `math_tool`、`image_tool`、`excel_python_tool`。
- `RESOURCE_NAME`：替換要授權及加解密的 Resource。
- `source`、`encrypted`：替換來源檔與加密輸出路徑。
- 保險庫密碼：由使用者在加密 Cell 設定並確認，解密 Cell 會重新詢問；scrypt 會從密碼與隨機 salt 派生 32-byte AES Key。
- `ROLE_PERMISSIONS`：可在 `course_utils.py` 調整教材角色對 Tool 與 Resource 的權限。

In [27]:
from pathlib import Path
import importlib, os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import course_utils
importlib.reload(course_utils)
from course_utils import *
print("教材根目錄：", ROOT)
connection = verify_cillm_key()
print("✅ API Key 驗證成功")
print("目前 Provider：", connection["provider"])
print("目前模型：", connection["model"])
print("首次回覆：", connection["reply"])

教材根目錄： C:\Users\rathe\Project\cillm\CILLM_Workshop\Lecture03
✅ API Key 驗證成功
目前 Provider： openai
目前模型： gpt-4o
首次回覆： API 連線成功


## 檢查目前 API Key 的基本 Scope

下方 Cell 會查詢並列出這支 Key 實際具備的 RBAC scopes。本教材呼叫 GPT-OSS 至少需要 `llm.chat`。

In [28]:
scopes = get_current_key_scopes()
if scopes is None:
    print("目前使用 OpenAI API；CILLM RBAC scope 不適用。")
else:
    print("目前 CILLM_API_KEY 具備的 scopes：")
    for scope in scopes:
        print("-", scope)
    required_scope = "llm.chat"
    print(f"✅ 已具備教材基本 scope：{required_scope}" if "*" in scopes or required_scope in scopes else f"❌ 缺少教材基本 scope：{required_scope}")

目前使用 OpenAI API；CILLM RBAC scope 不適用。


> **執行模式 Hint**
>
> - 本教材每章第一格會取得 `CILLM_API_KEY`；若 `.env` 未設定，Notebook 會以隱藏輸入提示使用者填入。
> - 取得 Key 後會立即呼叫一次 `openai/gpt-oss-120b`，確認 Key 與連線可用。
> - `CILLM_BASE_URL` 預設沿用 Lecture 02 的測試端點；缺少或無效 Key 時會立即停止。
> - CILLM 模式：文字使用 GPT-OSS、圖片使用 Gemma，兩者沿用相同 CILLM API key。
> - OpenAI 模式：文字與圖片都使用 `gpt-4o`，用來模擬 GPT-OSS 與 Gemma 的教材流程。

> **角色 Hint**
>
> - `employee`：Excel Python Tool 與機務 Resource 應被拒絕。
> - `operations`：可分析 Excel 與讀航務規範，但不可讀機務規範。
> - `maintenance`：可讀機務 Resource；設定正確 AES key 後預期解密成功。
> - `developer`：可用動態 Python Tool 並讀資安規範。
> - `admin`：全部通過；改錯 AES key 仍應解密失敗。

## RBAC 角色與題目對照

每次選擇一個 `CURRENT_ROLE`，再取消對應問題的註解。同一題也可故意換成其他 Role，觀察 RBAC 拒絕。

| Role | 允許的 Tool 重點 | 允許的 Resource 重點 |
|---|---|---|
| `guest` | `math_tool` | 公司與旅客公開資料 |
| `employee` | `math_tool` | 員工與旅客規範 |
| `operations` | math、image、Excel | 航務與旅客規範 |
| `maintenance` | math、image | 機務維修規範 |
| `developer` | math、image、Excel | 資安與員工規範 |
| `admin` | 全部 | 全部 |

In [29]:
# 預設：operations
CURRENT_ROLE = "operations"

# 每次只取消一行註解
# CURRENT_ROLE = "guest"
# CURRENT_ROLE = "employee"
# CURRENT_ROLE = "maintenance"
# CURRENT_ROLE = "developer"
# CURRENT_ROLE = "admin"

print("目前角色：", CURRENT_ROLE)
print("✅ allowed / ❌ denied 權限總覽：")
show(get_role_access_summary(CURRENT_ROLE))

目前角色： operations
✅ allowed / ❌ denied 權限總覽：
{
  "role": "operations",
  "tools": {
    "allowed": [
      "math_tool",
      "image_tool",
      "excel_python_tool"
    ],
    "denied": []
  },
  "resources": {
    "allowed": [
      "company_information",
      "passenger_service_rules",
      "flight_operations_guide"
    ],
    "denied": [
      "maintenance_guidelines",
      "it_security_policy",
      "employee_policy"
    ]
  }
}


## Tool 題庫：AI 選 Tool → Pydantic → RBAC → 實際執行

預設題與 `operations` 搭配。其他題目都在 Cell 內以註解保留。

In [30]:
# operations / developer / admin：Excel Tool
USER_REQUEST = "請精確統計 flight_delays.xlsx 中的航班總數。"

# guest / employee：math_tool 可通過
USER_REQUEST = "312 個座位乘以 87% 載客率，精確計算旅客數。"

# employee：故意測試 Excel Tool，預期被 RBAC 拒絕
USER_REQUEST = "請依部門計算 flight_delays.xlsx 的平均延誤。"

# maintenance / operations / developer / admin：image_tool
# USER_REQUEST = "請檢查 ramp_safety_inspection.png 中的機坪安全風險。"

# maintenance：故意測試 Excel Tool，預期被 RBAC 拒絕
# USER_REQUEST = "請找出 flight_delays.xlsx 延誤最久的航班。"

print("使用者問題：", USER_REQUEST)
tool_route = route_tool(USER_REQUEST)
print("Pydantic Tool Route："); show(tool_route.model_dump())
if tool_route.tool_name == "none": raise RuntimeError("AI 判斷不需要 Tool")
wanted_tool = tool_route.tool_name
allowed = authorize(CURRENT_ROLE, "tools", wanted_tool)

if not allowed:
    result = "權限不足：未執行 AI 選擇的 Tool"
elif wanted_tool == "math_tool":
    args = validate_tool_arguments(tool_route)
    result = math_tool(args.a, args.op, args.b)
elif wanted_tool == "image_tool":
    args = validate_tool_arguments(tool_route)
    image_path = find_named_data_file("images", USER_REQUEST, {".png", ".jpg", ".jpeg"})
    result = analyze_image(image_path, args.question)
elif wanted_tool == "excel_python_tool":
    validate_tool_arguments(tool_route)
    excel_path = find_named_data_file("excel", USER_REQUEST, {".xlsx", ".xlsm", ".xls"})
    generated = generate_excel_code(USER_REQUEST, excel_path)
    print("AI generated code：\n", generated.code)
    result = safe_excel_python(excel_path, generated.code)
else:
    result = f"未實作 {wanted_tool} 的執行器"

print_execution_trace(question=USER_REQUEST, role=CURRENT_ROLE, tool=wanted_tool, tool_auth="✓" if allowed else "✗", tool_result=result if allowed else "未執行", answer=result)

使用者問題： 請依部門計算 flight_delays.xlsx 的平均延誤。
Pydantic Tool Route：
{
  "tool_name": "excel_python_tool",
  "reason": "問題需要對 Excel 表格進行精確的統計和分組操作，以計算每個部門的平均延誤時間。",
  "arguments": {
    "question": "依部門計算平均延誤"
  }
}
AI generated code：
 df = pd.read_excel(excel_path)
result = df.groupby('department')['delay_minutes'].mean()
AI Agent 執行追蹤
使用者問題：
請依部門計算 flight_delays.xlsx 的平均延誤。

目前角色：
operations

是否需要工具：
excel_python_tool

工具權限檢查：
✓

工具執行結果：
department
地勤部    135.0
客服部     95.0
航務部    107.5
Name: delay_minutes, dtype: float64

最終回答：
department
地勤部    135.0
客服部     95.0
航務部    107.5
Name: delay_minutes, dtype: float64



## 使用者密碼 → scrypt → AES-256-GCM

AES-256 不直接使用人類密碼。本章會：

1. 請使用者設定密碼並再次確認。
2. 產生 16-byte 隨機 salt，用 scrypt 從密碼派生 32-byte AES Key。
3. 產生 12-byte 隨機 nonce，用 AES-256-GCM 建立保險庫。
4. 檔案只保存 salt、nonce、ciphertext 與 authentication tag，**不保存密碼或派生 Key**。
5. 解密 Cell 會重新詢問密碼；錯誤密碼會得到 `InvalidTag`。

In [32]:
# operations / admin：航務 Resource
USER_REQUEST = "雷雨警報時，航務人員需要重新評估哪些項目？"

# guest / employee / operations / admin：旅客 Resource
# USER_REQUEST = "航班延誤 120 分鐘應提供什麼協助？"

# employee / developer / admin：員工 Resource
# USER_REQUEST = "病假在什麼情況下需要補交證明？"

# maintenance / admin：機務 Resource
# USER_REQUEST = "維修工具短少時可以放行航機嗎？"

# developer / admin：資安 Resource
# USER_REQUEST = "API key 外洩時應採取哪些處置？"

# guest 故意測試機務 Resource，預期被 RBAC 拒絕
# USER_REQUEST = "請讀取機務維修規範的關鍵要求。"

print("使用者問題：", USER_REQUEST)
resource_route = route_resource(USER_REQUEST)
print("Pydantic Resource Route："); show(resource_route.model_dump())
if resource_route.resource_name == "none": raise RuntimeError("AI 判斷不需要 Resource")
RESOURCE_NAME = resource_route.resource_name
source = ROOT / "resources" / RESOURCE_CATALOG[RESOURCE_NAME][0]
vault_file = ROOT / "generated" / "password_vault" / f"{RESOURCE_NAME}.pw-aes256gcm"
resource_allowed = authorize(CURRENT_ROLE, "resources", RESOURCE_NAME)

if not resource_allowed:
    raise PermissionError(f"{CURRENT_ROLE} 沒有 {RESOURCE_NAME} 的存取權，不會建立保險庫。")

print("\n=== 加密前的來源明文摘要 ===")
plaintext = source.read_bytes().decode("utf-8")
print(plaintext[:300] + ("..." if len(plaintext) > 300 else ""))

print("\n=== 設定保險庫密碼 ===")
vault_password = prompt_new_vault_password()
encrypt_resource_with_password(source, vault_file, vault_password)
del vault_password  # 示範：加密後不在 Notebook 變數中保留密碼

print("\n=== 已建立密碼保護的 AES-256-GCM 保險庫 ===")
vault_evidence = inspect_password_vault(vault_file)
show(vault_evidence)
print("✅ 密碼未寫入檔案，派生的 AES Key 也未寫入檔案。")
print("請到下一格重新輸入密碼解鎖。")

使用者問題： 雷雨警報時，航務人員需要重新評估哪些項目？
Pydantic Resource Route：
{
  "resource_name": "flight_operations_guide",
  "reason": "使用者問題涉及雷雨警報和航務人員的評估，這與航班、航務、起飛、飛航計畫、雷雨與異常通報作業相關，因此選擇 flight_operations_guide 資源。"
}

=== 1. 加密前：來源明文摘要 ===
航班作業規範（虛構教學資料）
1. 起飛前須完成載重平衡確認。
2. 機長與簽派員共同確認飛航計畫。
3. 雷雨警報時須重新評估航路。
4. 延誤超過 30 分鐘通知地勤與客服。
5. 延誤達 120 分鐘啟動旅客服務協調。
6. 班機異常須記錄原因與時間軸。
7. 更換機型後須重新確認座位配置。
8. 油量低於計畫門檻須立即通報。
9. 關艙前完成旅客與行李核對。
10. 作業紀錄至少由當班主管覆核。
11. 天候惡化時應持續更新起飛時間、備降機場與燃油評估。
12. 航班延誤原因應使用核准分類，不得以未確認資訊對外說明。
13. 更換登機門後應同步通知地勤、客服與旅客資...
✅ 已取得正確秘密 Key（不顯示內容）

=== 2. 建立保險庫密文 ===
{
  "file": "C:\\Users\\rathe\\Project\\cillm\\CILLM_Workshop\\Lecture03\\generated\\encrypted_vault\\flight_operations_guide.aes256gcm",
  "format": "CILLM AES-256-GCM v1",
  "total_bytes": 971,
  "nonce_bytes": 12,
  "authentication_tag_bytes": 16,
  "sha256": "c789ad1cbbaf11bb37e4ce3b29c8c523ecfed26771f3ba69053f4ae782dd4f2d",
  "ciphertext_hex_preview": "fd3f069a29ad822cb1b9f23826a8296c2aea82bef6627f8ea34fa6e90a1eb11669f979d2f7c35f688e3

## 重新輸入密碼解鎖

這個 Cell 不知道剛才的密碼。輸入錯誤密碼時，GCM 認證會失敗；只有相同密碼才能從 salt 重建正確 AES Key 並還原明文。

In [ ]:
if "vault_file" not in globals() or not vault_file.exists():
    raise RuntimeError("請先執行上一格建立密碼保護的保險庫。")

unlock_password = prompt_vault_password()
try:
    decrypted_content = decrypt_resource_with_password(vault_file, unlock_password)
except Exception as error:
    decrypted_content = None
    print(f"❌ 解密失敗：{type(error).__name__}。密碼錯誤或密文遭竄改。")
finally:
    del unlock_password

if decrypted_content is not None:
    print("✅ 密碼正確，AES-256-GCM 解密與完整性驗證成功。")
    print("\n還原的明文摘要：")
    print(decrypted_content[:300] + ("..." if len(decrypted_content) > 300 else ""))
    final_answer = ask_gpt_oss(USER_REQUEST, decrypted_content, "只能根據成功解密的 Resource 回答。使用繁體中文。")
    print_execution_trace(question=USER_REQUEST, role=CURRENT_ROLE, resource=RESOURCE_NAME, resource_auth="✓", aes="密碼正確，解密成功", tool_result=vault_evidence, answer=final_answer)

RBAC 決定誰能存取；AES-256 保護被複製的檔案。沒有權限時，程式不會先解密。

### 小練習

切換 `CURRENT_ROLE`，或改用錯誤 AES key，重新執行並比較追蹤。